# Online Retail Customer Churn Prediction

End-to-end churn prediction pipeline covering:
- Data loading & cleaning
- Exploratory data analysis (EDA)
- Feature engineering
- Model training (Logistic Regression & Random Forest)
- Evaluation (ROC AUC, classification report)
- Feature importance interpretation

## 1 – Imports and data load

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Allow importing from src/ when running the notebook from the project root
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

sns.set_theme(style='whitegrid')

In [ ]:
data_path = Path('../data/raw/online_retail_churn_raw.csv')
df = pd.read_csv(data_path)

print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print('Missing values per column:')
print(df.isna().sum())

print('\nChurn rate:')
print(df['Churn'].value_counts(normalize=True).round(3))

## 2 – Exploratory Data Analysis (EDA)

In [ ]:
# Churn vs. non-churn counts
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='Churn', data=df, palette='Set2', ax=ax)
ax.set_title('Churn vs. Non-Churn')
ax.set_xticklabels(['Non-Churn (0)', 'Churn (1)'])
plt.tight_layout()
plt.show()

In [ ]:
# Numeric feature distributions
num_cols = [
    'Age', 'Annual_Income_USD', 'Spending_Score', 'Total_Purchases',
    'Avg_Purchase_Value', 'Satisfaction_Score',
    'Website_Visits_Last_Month', 'Avg_Time_Per_Visit_Minutes',
    'Support_Tickets_Last_6_Months', 'Referred_Friends',
]

df[num_cols].describe().T.round(2)

In [ ]:
# Histograms coloured by churn status
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    for churn_val, label in [(0, 'Non-Churn'), (1, 'Churn')]:
        axes[i].hist(df.loc[df['Churn'] == churn_val, col],
                     bins=30, alpha=0.6, label=label, density=True)
    axes[i].set_title(col, fontsize=9)
    axes[i].legend(fontsize=7)
plt.suptitle('Feature Distributions by Churn Status', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(
    df[num_cols + ['Churn']].corr(),
    annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.4
)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Churn rate by categorical features
cat_cols = ['Membership_Status', 'Preferred_Payment_Method', 'Region', 'Gender']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(18, 5))
for ax, col in zip(axes, cat_cols):
    churn_rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    churn_rate.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(f'Churn Rate by {col}', fontsize=10)
    ax.set_ylabel('Churn Rate')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 3 – Feature Engineering

In [ ]:
from src.data_prep import clean
from src.features import add_features

df_clean = clean(df)
df_feat = add_features(df_clean)

print('New features added:')
new_cols = [c for c in df_feat.columns if c not in df_clean.columns]
print(new_cols)
df_feat[new_cols].describe().T.round(2)

## 4 – Model Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from src.features import build_preprocessor

TARGET_COL = 'Churn'
X = df_feat.drop(columns=[TARGET_COL])
y = df_feat[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Train size:', len(X_train), '  Test size:', len(X_test))
print('Train churn rate:', y_train.mean().round(3))

In [ ]:
# Logistic Regression
logreg_pipeline = Pipeline([
    ('preprocess', build_preprocessor(X_train)),
    ('model', LogisticRegression(max_iter=500, class_weight='balanced')),
])
logreg_pipeline.fit(X_train, y_train)

y_pred_lr = logreg_pipeline.predict(X_test)
y_proba_lr = logreg_pipeline.predict_proba(X_test)[:, 1]

print('Logistic Regression  ROC AUC:', roc_auc_score(y_test, y_proba_lr).round(4))
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Random Forest
rf_pipeline = Pipeline([
    ('preprocess', build_preprocessor(X_train)),
    ('model', RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=42
    )),
])
rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

print('Random Forest  ROC AUC:', roc_auc_score(y_test, y_proba_rf).round(4))
print(classification_report(y_test, y_pred_rf))

## 5 – Evaluation

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
RocCurveDisplay.from_predictions(y_test, y_proba_lr, name='Logistic Regression', ax=ax)
RocCurveDisplay.from_predictions(y_test, y_proba_rf, name='Random Forest', ax=ax)
ax.plot([0, 1], [0, 1], '--', color='gray', label='Random Chance')
ax.set_title('ROC Curves – Churn Models')
ax.legend()
plt.tight_layout()
plt.show()

## 6 – Feature Importance

In [ ]:
# Reconstruct feature names from the fitted pipeline
preprocessor = rf_pipeline.named_steps['preprocess']
num_names = preprocessor.transformers_[0][2]
cat_encoder = preprocessor.transformers_[1][1]
cat_names = cat_encoder.get_feature_names_out(preprocessor.transformers_[1][2]).tolist()
feature_names = list(num_names) + cat_names

importances = pd.Series(
    rf_pipeline.named_steps['model'].feature_importances_,
    index=feature_names
).nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top-20 Feature Importances – Random Forest')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()